# Chatbot RAG - Asistente de Compras Maincal

Asistente de compras basado en **LLM + RAG** para el producto **Cronos-N04**.

**Arquitectura de datos (3 fuentes):**
1. **Cerco de proveedores** (PDF) -> se trocea en *chunks* y se busca por similitud semantica (embeddings).
2. **BOM** (xlsx, salida del ERP) -> se carga con pandas y se **inyecta exacta** en el contexto (no pasa por embeddings).
3. **Stock actual** -> input manual (salida de Logistica: ERP/CNN); se **inyecta exacto** en el contexto.

El contexto final que recibe el modelo = *chunks del PDF* + *BOM* + *stock actual*.

---

**Como correr en VS Code (local):**
1. Tener en la **misma carpeta** que este notebook: `Cerco_informacion_Maincal.pdf` y `BOM_Cronos-N04.xlsx`.
2. Elegir el kernel de **Python 3** (boton "Select Kernel" arriba a la derecha).
3. Ejecutar las celdas **en orden**, de arriba hacia abajo.
4. En el PASO 1 se pide la **API key** de OpenAI: pegarla en la cajita que aparece arriba.

> Si recien instalaste las librerias, puede hacer falta **reiniciar el kernel**
> (boton "Restart") para que las tome.
>
> Tambien funciona en **Google Colab**: la API key se toma de Secrets o se pide por pantalla.

In [ ]:
# =============================================================
# PASO 0 - INSTALACION DE DEPENDENCIAS
# =============================================================
# En Google Colab conviene ejecutar esta celda la primera vez.
# En local (Cursor/VS Code) podes saltearla si ya tenes los paquetes.
%pip install -q openai pypdf pandas openpyxl scikit-learn numpy

In [ ]:
# =============================================================
# PASO 1 - IMPORTS Y CARGA DE LA API KEY
# =============================================================
import os
import getpass
import numpy as np
import pandas as pd
from pypdf import PdfReader
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI


def cargar_api_key():
    """Obtiene la API key de OpenAI de forma flexible:
    1) Google Colab Secrets (recomendado en Colab).
    2) Variable de entorno OPENAI_API_KEY (recomendado en local).
    3) Prompt manual seguro (getpass) como ultimo recurso.
    """
    # 1) Google Colab Secrets
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("OPENAI_API_KEY")
        if key:
            print("API key cargada desde Colab Secrets.")
            return key
    except Exception:
        pass

    # 2) Variable de entorno
    key = os.environ.get("OPENAI_API_KEY")
    if key:
        print("API key cargada desde variable de entorno.")
        return key

    # 3) Prompt manual
    print("No se encontro la API key automaticamente.")
    return getpass.getpass("Pega tu API key de OpenAI: ").strip()


api_key = cargar_api_key()
if not api_key:
    raise ValueError("No se proporciono una API key valida.")

client = OpenAI(api_key=api_key)
print("Cliente OpenAI configurado correctamente.")

In [ ]:
# =============================================================
# PASO 2 - PARAMETROS DE CONFIGURACION
# =============================================================
# Archivos de entrada (deben estar en la misma carpeta)
PDF_PATH = "Cerco_informacion_Maincal.pdf"   # cerco de proveedores
BOM_PATH = "BOM_Cronos-N04.xlsx"             # BOM (salida del ERP)

# Modelos de OpenAI
MODELO_EMBEDDING = "text-embedding-3-small"
MODELO_CHAT = "gpt-4o-mini"

# Parametros del troceado (chunking)
TAM_CHUNK = 1500     # tamano de cada chunk en caracteres
OVERLAP_CHUNK = 300  # solapamiento entre chunks (~20%)

# Parametros de recuperacion (retrieval)
TOP_K = 3            # cantidad de chunks mas relevantes a recuperar

# Parametros de generacion del LLM
# temperatura: 0.0-0.2 -> respuestas deterministas y fieles a los datos.
# Para un asistente que debe responder con datos exactos, conviene bajo.
TEMPERATURA = 0.1
MAX_TOKENS = 800    # largo maximo de la respuesta generada
TOP_P = 1.0         # se deja por defecto (la temperatura ya controla la aleatoriedad)

print("Parametros configurados.")

In [ ]:
# =============================================================
# PASO 3 - LEER EL PDF DEL CERCO DE PROVEEDORES
# =============================================================
if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(
        f"No se encontro '{PDF_PATH}'. Subi el PDF a la carpeta del notebook.")

reader = PdfReader(PDF_PATH)
paginas = []          # texto de cada pagina por separado (1 ficha por pagina)
for page in reader.pages:
    contenido = page.extract_text()
    if contenido and contenido.strip():
        paginas.append(contenido.strip())

texto = "\n".join(paginas)  # texto completo (solo informativo)

if not texto.strip():
    raise ValueError(
        "El PDF se leyo pero no se pudo extraer texto. "
        "Puede ser un PDF escaneado (imagen) sin texto seleccionable.")

print(f"Texto extraido: {len(texto):,} caracteres en {len(paginas)} paginas.")

In [ ]:
# =============================================================
# PASO 4 - TROCEAR EN CHUNKS (1 chunk por ficha/pagina)
# =============================================================
# El cerco tiene una ficha de proveedor por pagina. Lo natural aca es que
# cada ficha sea un chunk independiente: asi un insumo no se mezcla con otro
# y la recuperacion devuelve la ficha completa del insumo consultado.
# Si alguna pagina fuese mas larga que TAM_CHUNK, se subdivide con solapamiento.
def split_largo(texto, chunk_size, overlap):
    partes = []
    start = 0
    while start < len(texto):
        end = start + chunk_size
        partes.append(texto[start:end])
        start = end - overlap
    return partes


chunks = []
for pag in paginas:
    if len(pag) <= TAM_CHUNK:
        chunks.append(pag)               # la ficha entra completa en un chunk
    else:
        chunks.extend(split_largo(pag, TAM_CHUNK, OVERLAP_CHUNK))

print(f"Texto dividido en {len(chunks)} chunks (1 por ficha de proveedor).")

In [ ]:
# =============================================================
# PASO 5 - GENERAR EMBEDDINGS DE LOS CHUNKS
# =============================================================
BATCH_SIZE = 100  # procesar por lotes evita limites de la API

def get_embeddings(texts, batch_size=BATCH_SIZE):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        lote = texts[i:i + batch_size]
        response = client.embeddings.create(model=MODELO_EMBEDDING, input=lote)
        for item in response.data:
            embeddings.append(item.embedding)
    return np.array(embeddings)


try:
    chunk_embeddings = get_embeddings(chunks)
    print(f"Embeddings generados: {chunk_embeddings.shape}")
except Exception as e:
    print("ERROR al llamar a la API de OpenAI.")
    print("Causas mas comunes:")
    print("  - API key invalida o mal pegada -> volve a correr el PASO 1.")
    print("  - Sin saldo/credito en la cuenta -> cargar en platform.openai.com/billing")
    print()
    print(f"Detalle tecnico: {e}")
    raise

In [ ]:
# =============================================================
# PASO 6 - DATOS NUMERICOS: BOM (xlsx) + STOCK ACTUAL (input manual)
# =============================================================
# Estos datos NO pasan por embeddings: se inyectan EXACTOS en el contexto
# para garantizar precision numerica (cantidades, stock).

# --- 6.a) BOM desde el ERP (xlsx) ---
# La cabecera real esta en la fila 3 del archivo (header=2).
bom_df = pd.read_excel(BOM_PATH, header=2)
# Conservar solo filas validas (descarta notas al pie sin cantidad)
bom_df = bom_df.dropna(subset=["insumo", "cantidad_por_par"])
print("BOM cargada:")
print(bom_df.to_string(index=False))

# --- 6.b) STOCK ACTUAL (salida de Logistica: ERP / CNN) ---
# EDITAR A MANO para la demo. Solo se cargan los 3 insumos criticos;
# el resto de los insumos se considera con stock suficiente (no limita).
stock_actual = {
    "Conjunto Sistema PU": 1200,          # kg
    "Puntera de Acero 59 Normal": 18000,  # unidades
    "Caja de empaque": 5000,              # unidades
}

# --- 6.c) Armar el bloque de texto numerico que se inyecta al contexto ---
def construir_datos_numericos(bom_df, stock_actual):
    lineas = ["=== BOM (consumo por par del producto Cronos-N04) ==="]
    for _, fila in bom_df.iterrows():
        critico = " [CRITICO]" if str(fila.get("critico", "")).upper() == "SI" else ""
        lineas.append(
            f"- {fila['insumo']}: {fila['cantidad_por_par']} {fila['unidad']} por par{critico}")
    lineas.append("")
    lineas.append("=== STOCK ACTUAL de insumos criticos (fuente: ERP/CNN) ===")
    for insumo, cant in stock_actual.items():
        lineas.append(f"- {insumo}: {cant}")
    lineas.append("")
    lineas.append("Nota: los insumos no listados en el stock se consideran con "
                  "disponibilidad suficiente y no son limitantes.")
    return "\n".join(lineas)


datos_numericos = construir_datos_numericos(bom_df, stock_actual)
print("\n" + datos_numericos)

In [ ]:
# =============================================================
# PASO 7 - BUSQUEDA SEMANTICA (RETRIEVE)
# =============================================================
def search_similar_chunks(question, chunk_embeddings, chunks, k=TOP_K):
    question_embedding = get_embeddings([question])[0]
    similarities = cosine_similarity([question_embedding], chunk_embeddings)[0]
    top_indices = similarities.argsort()[::-1][:k]
    resultados = []
    for idx in top_indices:
        resultados.append({
            "index": int(idx),
            "score": float(similarities[idx]),
            "chunk": chunks[idx],
        })
    return resultados

In [ ]:
# =============================================================
# PASO 8 - GENERAR RESPUESTA CON RAG (AUGMENT + GENERATE)
# =============================================================
SYSTEM_PROMPT = """
Eres un asistente de compras de la empresa Maincal (fabrica de calzado de seguridad industrial).
Tu funcion es ayudar a decidir prioridades de compra del producto Cronos-N04, basandote
UNICAMENTE en la informacion del contexto (fichas de proveedores + BOM + stock actual).

REGLAS GENERALES:
- Responde solo con la informacion del contexto. Si un dato no esta, responde:
  "No tengo esa informacion disponible". Nunca inventes datos ni supongas valores.
- NO inventes cantidades de orden. Si la pregunta no menciona una cantidad de pares,
  no asumas ninguna.

COMO EVALUAR PRIORIDAD (cuando se pregunta que insumo priorizar):
- Para cada insumo critico, compara su stock actual contra su stock minimo (de la ficha).
  Si el stock actual esta por debajo del stock minimo, ese insumo debe priorizarse.
- Entre los que esten por debajo del minimo, ordena por riesgo: mayor lead time,
  proveedor unico, importacion o paralizacion de linea = mas urgente.
- Si el stock actual esta por encima del stock minimo, no requiere atencion inmediata.
- No uses ordenes hipoteticas para decidir prioridad; usa stock actual vs stock minimo.

COMO EVALUAR UNA ORDEN PUNTUAL (solo si la pregunta indica una cantidad de pares):
- Calcula necesidad = cantidad_de_pares x consumo_por_par (de la BOM) para cada insumo.
- Compara la necesidad contra el stock actual e indica si alcanza o si hay riesgo de quiebre.

Se claro y concreto, y explica el porque (stock vs minimo, lead time, proveedor unico,
importacion, etc.).
"""

def rag_answer(question, chunk_embeddings, chunks, k=TOP_K, max_tokens=MAX_TOKENS):
    # RETRIEVE: chunks mas relevantes del cerco de proveedores (PDF)
    resultados = search_similar_chunks(question, chunk_embeddings, chunks, k)
    contexto_proveedores = "\n\n".join(r["chunk"] for r in resultados)

    # AUGMENT: contexto = chunks del PDF + datos numericos (BOM + stock)
    contexto = (
        "--- FICHAS DE PROVEEDORES (recuperadas del cerco) ---\n"
        f"{contexto_proveedores}\n\n"
        "--- DATOS NUMERICOS (BOM + STOCK ACTUAL) ---\n"
        f"{datos_numericos}"
    )

    user_prompt = f"""
    CONTEXTO:
    {contexto}

    PREGUNTA: {question}

    RESPUESTA:
    """

    # GENERATE
    response = client.chat.completions.create(
        model=MODELO_CHAT,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=TEMPERATURA,
        max_tokens=max_tokens,
        top_p=TOP_P,
    )
    return response.choices[0].message.content


# -------------------------------------------------------------
# VERSION CON MEMORIA (para preguntas de seguimiento)
# -------------------------------------------------------------
# Mantiene el historial de la conversacion para entender repreguntas
# como "y cual es su proveedor?" (sabe que "su" se refiere a lo anterior).
historial = []  # lista de turnos {"role": ..., "content": ...}


def reset_historial():
    """Borra la memoria de la conversacion (empezar de cero)."""
    global historial
    historial = []
    print("Memoria de conversacion reiniciada.")


def rag_chat(question, k=TOP_K, max_tokens=MAX_TOKENS):
    """Igual que rag_answer pero RECORDANDO los turnos anteriores."""
    global historial

    # RETRIEVE + AUGMENT (contexto siempre fresco con los datos actuales)
    resultados = search_similar_chunks(question, chunk_embeddings, chunks, k)
    contexto_proveedores = "\n\n".join(r["chunk"] for r in resultados)
    contexto = (
        "--- FICHAS DE PROVEEDORES (recuperadas del cerco) ---\n"
        f"{contexto_proveedores}\n\n"
        "--- DATOS NUMERICOS (BOM + STOCK ACTUAL) ---\n"
        f"{datos_numericos}"
    )
    user_prompt = f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {question}"

    # Mensajes = system + historial previo + nueva pregunta
    mensajes = [{"role": "system", "content": SYSTEM_PROMPT}]
    mensajes += historial
    mensajes.append({"role": "user", "content": user_prompt})

    response = client.chat.completions.create(
        model=MODELO_CHAT, messages=mensajes,
        temperature=TEMPERATURA, max_tokens=max_tokens, top_p=TOP_P,
    )
    answer = response.choices[0].message.content

    # Guardamos en el historial SOLO la pregunta y la respuesta (sin el
    # contexto gigante), para que la memoria quede liviana y clara.
    historial.append({"role": "user", "content": question})
    historial.append({"role": "assistant", "content": answer})
    return answer

In [ ]:
# =============================================================
# PASO 9 - PREGUNTAS DE EJEMPLO (util para la defensa / demo)
# =============================================================
preguntas_ejemplo = [
    "Que insumo deberia priorizar esta semana?",
    "Con una orden de 1000 pares del Cronos-N04, tenemos stock suficiente?",
    "Cual es el lead time y el proveedor del Conjunto Sistema PU?",
]

for pregunta in preguntas_ejemplo:
    print(f"PREGUNTA: {pregunta}")
    respuesta = rag_answer(pregunta, chunk_embeddings, chunks, k=TOP_K)
    print(f"RESPUESTA:\n{respuesta}\n")
    print("-" * 70)

In [ ]:
# =============================================================
# PASO 10 - MODO INTERACTIVO (opcional)
# =============================================================
# OJO: este modo abre un bucle que espera preguntas. Para que "Run All" no
# se quede trabado, NO se ejecuta solo: se define como funcion y la llamas
# cuando quieras usarlo.
def modo_interactivo():
    # Empezamos con la memoria limpia para esta conversacion.
    reset_historial()
    print("Asistente de compras Maincal. Escribi 'salir' para terminar.\n")
    while True:
        pregunta = input("Tu pregunta: ").strip()
        if pregunta.lower() in ("salir", "exit", "quit", ""):
            print("Hasta luego.")
            break
        # Usamos rag_chat (CON memoria) para entender repreguntas como
        # "y cual es su proveedor?".
        respuesta = rag_chat(pregunta, k=TOP_K)
        print(f"\nRespuesta:\n{respuesta}\n")
        print("-" * 70)


# Para iniciar el modo interactivo, quita el # de la linea de abajo y ejecuta esta celda:
# modo_interactivo()

## Celda de DEMO: cambiar stock en vivo y preguntar

Pensada para la defensa. Permite **cambiar el stock de un insumo crítico y preguntar en un solo paso**, sin tocar el Paso 6 ni regenerar embeddings (es instantáneo y no consume API de embeddings).

In [ ]:
# =============================================================
# DEMO - Cambiar stock en vivo y preguntar
# =============================================================
def preguntar_con_stock(pregunta, nuevo_stock=None):
    """Actualiza (opcionalmente) el stock de los insumos criticos y responde.
    No recalcula embeddings: solo reconstruye el bloque numerico inyectable.

    Ejemplo:
        preguntar_con_stock(
            "Que insumo deberia priorizar esta semana?",
            nuevo_stock={"Conjunto Sistema PU": 500},
        )
    """
    global datos_numericos
    if nuevo_stock:
        stock_actual.update(nuevo_stock)
        datos_numericos = construir_datos_numericos(bom_df, stock_actual)

    print("Stock actual usado en esta consulta:")
    for insumo, cant in stock_actual.items():
        print(f"  - {insumo}: {cant}")
    print("-" * 70)
    print(rag_answer(pregunta, chunk_embeddings, chunks))


# --- Ejemplo: bajar el PU a 500 kg y ver como cambia la prioridad ---
preguntar_con_stock(
    "Que insumo deberia priorizar esta semana?",
    nuevo_stock={"Conjunto Sistema PU": 500},
)